In [8]:
import pandas as pd
import numpy as np
df = pd.read_csv("df_merged.csv")
customers_features = pd.read_csv("customer_segments.csv")
df.columns


Index(['SalesID', 'SalesPersonID', 'CustomerID', 'ProductID', 'Quantity',
       'Discount', 'TotalPrice', 'SalesDate', 'TransactionNumber', 'Price',
       'Revenue', 'FirstName', 'MiddleInitial', 'LastName', 'CityID',
       'Address', 'ProductName', 'CategoryID', 'Class', 'Resistant',
       'IsAllergic', 'VitalityDays', 'CategoryName', 'CityName', 'CountryID',
       'CountryName', 'year', 'month', 'day', 'day_of_week', 'quarter', 'week',
       'season'],
      dtype='str')

In [9]:
customer_category = (
    df.groupby(["CustomerID", "CategoryName"])
      .agg(
          category_spent=("Revenue", "sum"),
          category_items=("Quantity", "sum"),
          category_transactions=("TransactionNumber", "nunique")
      )
      .reset_index()
)

customer_category.head()

customer_category.to_csv("customer_category_summary.csv", index=False)



In [10]:
favorite_category = (
    customer_category
    .sort_values(["CustomerID", "category_spent"], ascending=[True, False])
    .groupby("CustomerID")
    .head(1)
    .rename(columns={
        "CategoryName": "favorite_category",
        "category_spent": "favorite_category_spent",
        "category_items": "favorite_category_items",
        "category_transactions": "favorite_category_transactions"
    })
)

favorite_category.head()

,CustomerID,favorite_category,favorite_category_spent,favorite_category_items,favorite_category_transactions
4,1,Grain,529.24417,8,8
13,2,Confections,763.13080,12,12
22,3,Beverages,492.07604,12,12
38,4,Meat,541.53418,9,9
45,5,Cereals,705.20450,14,14


In [11]:
customers_marketing = customers_features.merge(
    favorite_category[
        [
            "CustomerID",
            "favorite_category",
            "favorite_category_spent",
            "favorite_category_items",
            "favorite_category_transactions"
        ]
    ],
    left_on="customer_id",
    right_on="CustomerID",
    how="left"
)

customers_marketing.head()



,customer_id,total_spent,avg_check,num_checks,total_items,avg_items_per_check,avg_unique_products,recency,cluster,segment_name,CustomerID,favorite_category,favorite_category_spent,favorite_category_items,favorite_category_transactions
0,1,3135.08003,49.763175,63,63,1.0,1.0,4,2,Low value customers,1,Grain,529.24417,8,8
1,2,3355.04483,53.254680,63,63,1.0,1.0,6,2,Low value customers,2,Confections,763.13080,12,12
2,3,3318.65282,47.409326,70,70,1.0,1.0,1,2,Low value customers,3,Beverages,492.07604,12,12
3,4,3122.56073,45.254503,69,69,1.0,1.0,2,2,Low value customers,4,Meat,541.53418,9,9
4,5,2650.34531,44.921107,59,59,1.0,1.0,1,2,Low value customers,5,Cereals,705.20450,14,14


In [4]:
def recommend_campaign(segment, favorite_category):
    if pd.isna(favorite_category):
        return "General promotion campaign"
    
    if segment == "VIP customers":
        return f"Premium offer in {favorite_category} + loyalty bonus"
    
    elif segment == "Regular customers":
        return f"Bundle offer or cross-sell campaign in {favorite_category}"
    
    elif segment == "At-risk customers":
        return f"Reactivation discount in {favorite_category}"
    
    elif segment == "Low value customers":
        return f"Introductory discount for popular products in {favorite_category}"
    
    else:
        return f"Personalized promotion in {favorite_category}"

In [5]:
customers_marketing["marketing_recommendation"] = customers_marketing.apply(
    lambda row: recommend_campaign(row["segment_name"], row["favorite_category"]),
    axis=1
)

customers_marketing[
    ["customer_id", "segment_name", "favorite_category", "marketing_recommendation"]
].head(10)

,customer_id,segment_name,favorite_category,marketing_recommendation
0,1,Low value customers,Grain,Introductory discount for popular products in ...
1,2,Low value customers,Confections,Introductory discount for popular products in ...
2,3,Low value customers,Beverages,Introductory discount for popular products in ...
3,4,Low value customers,Meat,Introductory discount for popular products in ...
4,5,Low value customers,Cereals,Introductory discount for popular products in ...
5,6,Low value customers,Snails,Introductory discount for popular products in ...
6,7,At-risk customers,Confections,Reactivation discount in Confections
7,8,Low value customers,Confections,Introductory discount for popular products in ...
8,9,Low value customers,Shell fish,Introductory discount for popular products in ...
9,10,Low value customers,Confections,Introductory discount for popular products in ...


In [6]:
customers_marketing.groupby(["segment_name", "favorite_category"]).size().reset_index(name="customers_count").sort_values(
    ["segment_name", "customers_count"],
    ascending=[True, False]
).head(20)

customers_marketing.to_csv("customer_marketing_recommendations.csv", index=False)

###  marketing recommendations

На этом этапе была добавлена логика персонализированных маркетинговых рекомендаций.

Для каждого клиента была определена любимая категория товаров на основе суммы покупок в каждой категории. Затем эта информация была объединена с результатами клиентской сегментации.
